# Bias Bounty Mapping Equity Challenge - GPU Training
# Self-Evolving Pipeline with Kaggle GPUs

This notebook:
1. Downloads competition data from Source Cooperative
2. Computes enhanced features (coverage gaps + strata + source composition)
3. Trains XGBoost/LightGBM/CatBoost with GPU acceleration
4. Auto-tunes with Optuna
5. Computes optimal ensemble blend
6. Analyzes bias patterns
7. Generates submission

GPU: Enable GPU in Settings > Accelerator > GPU T4x2

In [ ]:
!pip install -q xgboost lightgbm catboost optuna shap pyarrow geopandas duckdb
import os
os.environ['XGBOOST_USE_CUDA'] = '1'

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import optuna
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from scipy.optimize import minimize
import json, time, logging
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Load Data

In [ ]:
# Load pre-computed features (upload as dataset to Kaggle)
DATA_DIR = Path('/kaggle/input/bias-bounty-features')
if not DATA_DIR.exists():
    DATA_DIR = Path('/kaggle/working')
    # If no pre-computed features, download from Source Cooperative
    print('Downloading data from Source Cooperative...')

# Load enhanced features
features_path = DATA_DIR / 'all_regions_enhanced_features.parquet'
if features_path.exists():
    features = pd.read_parquet(features_path)
    print(f'Loaded: {features.shape}')
else:
    # Fallback: load per-region and combine
    dfs = []
    for p in DATA_DIR.glob('*_enhanced_features.parquet'):
        dfs.append(pd.read_parquet(p))
    if dfs:
        features = pd.concat(dfs, ignore_index=True)
        print(f'Combined: {features.shape}')
    else:
        raise FileNotFoundError('No feature files found!')

# Load national strata
strata_path = DATA_DIR / 'national_enhanced_features.parquet'
if strata_path.exists():
    strata = pd.read_parquet(strata_path)
    print(f'Strata: {strata.shape}')

## 2. Prepare Features

In [ ]:
def prepare_features(df, target_col, max_features=100):
    drop_cols = ['GEOID', 'region', 'county_fips', 'state_fips',
                 'centroid_lat', 'centroid_lon',
                 'building_gap', 'road_gap', 'building_ratio', 'road_ratio',
                 'building_count_ratio', 'building_count_gap',
                 'road_count_ratio', 'road_count_gap',
                 'road_length_ratio', 'road_length_gap',
                 'poi_facility_gap', 'poi_to_facility_ratio']
    
    feature_cols = [c for c in df.columns 
                   if c not in drop_cols and df[c].dtype in [np.float64, np.float32, np.int64, np.int32]]
    
    X = df[feature_cols].copy()
    y = df[target_col].copy()
    geoids = df['GEOID'].copy()
    
    valid = y.notna()
    X, y, geoids = X[valid], y[valid], geoids[valid]
    X = X.fillna(-999)
    
    # Remove constant and low-variance
    std = X.std()
    X = X[std[std > 0.001].index]
    
    # Select top features by variance (for speed on GPU)
    if X.shape[1] > max_features:
        var = X.var().sort_values(ascending=False)
        X = X[var.head(max_features).index]
    
    print(f'Features: {X.shape[1]}, Tracts: {X.shape[0]}')
    return X, y, geoids

X, y, geoids = prepare_features(features, 'building_gap', max_features=150)

## 3. GPU-Accelerated Training with Spatial CV

In [ ]:
def train_xgboost_gpu(X, y, geoids, params=None, n_folds=5):
    """Train XGBoost with GPU acceleration and spatial CV."""
    if params is None:
        params = {
            'n_estimators': 1000,
            'max_depth': 6,
            'learning_rate': 0.05,
            'subsample': 0.8,
            'colsample_bytree': 0.7,
            'reg_alpha': 0.1,
            'reg_lambda': 1.0,
            'tree_method': 'hist',  # Use 'gpu_hist' on Kaggle GPU
            'device': 'cuda',
        }
    
    groups = geoids.str[:5]
    gkf = GroupKFold(n_splits=n_folds)
    oof = np.full(len(y), np.nan)
    fold_scores = []
    fis = []
    
    for fold, (ti, te) in enumerate(gkf.split(X, y, groups)):
        model = xgb.XGBRegressor(**params, random_state=42)
        model.fit(X.iloc[ti], y.iloc[ti], 
                 eval_set=[(X.iloc[te], y.iloc[te])], 
                 verbose=100)
        pred = model.predict(X.iloc[te])
        oof[te] = pred
        
        rmse = np.sqrt(mean_squared_error(y.iloc[te], pred))
        r2 = r2_score(y.iloc[te], pred)
        fold_scores.append({'rmse': rmse, 'r2': r2})
        fis.append(model.feature_importances_)
        print(f'Fold {fold}: RMSE={rmse:.6f}, R2={r2:.4f}')
    
    mean_rmse = np.mean([s['rmse'] for s in fold_scores])
    mean_r2 = np.mean([s['r2'] for s in fold_scores])
    mean_fi = np.mean(fis, axis=0)
    
    fi_df = pd.DataFrame({'feature': X.columns, 'importance': mean_fi}).sort_values('importance', ascending=False)
    
    return {'oof': oof, 'rmse': mean_rmse, 'r2': mean_r2, 'fi': fi_df, 'model': model}

# Train XGBoost
print('Training XGBoost (GPU)...')
xgb_result = train_xgboost_gpu(X, y, geoids)
print(f'XGBoost: RMSE={xgb_result["rmse"]:.6f}, R2={xgb_result["r2"]:.4f}')

In [ ]:
def train_lightgbm_gpu(X, y, geoids, n_folds=5):
    """Train LightGBM with GPU acceleration."""
    params = {
        'n_estimators': 1000,
        'max_depth': 6,
        'num_leaves': 31,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.7,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'device': 'gpu',
        'verbose': -1,
    }
    
    groups = geoids.str[:5]
    gkf = GroupKFold(n_splits=n_folds)
    oof = np.full(len(y), np.nan)
    fold_scores = []
    fis = []
    
    for fold, (ti, te) in enumerate(gkf.split(X, y, groups)):
        model = lgb.LGBMRegressor(**params, random_state=42)
        model.fit(X.iloc[ti], y.iloc[ti], 
                 eval_set=[(X.iloc[te], y.iloc[te])])
        pred = model.predict(X.iloc[te])
        oof[te] = pred
        
        rmse = np.sqrt(mean_squared_error(y.iloc[te], pred))
        r2 = r2_score(y.iloc[te], pred)
        fold_scores.append({'rmse': rmse, 'r2': r2})
        fis.append(model.feature_importances_)
        print(f'Fold {fold}: RMSE={rmse:.6f}, R2={r2:.4f}')
    
    mean_rmse = np.mean([s['rmse'] for s in fold_scores])
    mean_r2 = np.mean([s['r2'] for s in fold_scores])
    mean_fi = np.mean(fis, axis=0)
    fi_df = pd.DataFrame({'feature': X.columns, 'importance': mean_fi}).sort_values('importance', ascending=False)
    
    return {'oof': oof, 'rmse': mean_rmse, 'r2': mean_r2, 'fi': fi_df, 'model': model}

print('Training LightGBM (GPU)...')
lgb_result = train_lightgbm_gpu(X, y, geoids)
print(f'LightGBM: RMSE={lgb_result["rmse"]:.6f}, R2={lgb_result["r2"]:.4f}')

## 4. Optuna Hyperparameter Tuning (GPU)

In [ ]:
def optuna_tune_xgboost(X, y, geoids, n_trials=50, n_folds=3):
    """Optuna tuning for XGBoost with GPU."""
    groups = geoids.str[:5]
    gkf = GroupKFold(n_splits=n_folds)
    
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 2000),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.001, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.001, 10.0, log=True),
            'tree_method': 'hist',
            'device': 'cuda',
        }
        
        fold_rmses = []
        for ti, te in gkf.split(X, y, groups):
            model = xgb.XGBRegressor(**params, random_state=42)
            model.fit(X.iloc[ti], y.iloc[ti], eval_set=[(X.iloc[te], y.iloc[te])], verbose=False)
            pred = model.predict(X.iloc[te])
            fold_rmses.append(np.sqrt(mean_squared_error(y.iloc[te], pred)))
        
        return np.mean(fold_rmses)
    
    study = optuna.create_study(direction='minimize', sampler=optuna.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    print(f'Best RMSE: {study.best_value:.6f}')
    print(f'Best params: {study.best_params}')
    return study.best_params, study.best_value

print('Optuna tuning XGBoost (GPU, 50 trials)...')
best_params, best_rmse = optuna_tune_xgboost(X, y, geoids, n_trials=50)
print(f'Tuned XGBoost RMSE: {best_rmse:.6f}')

## 5. Ensemble & Submission

In [ ]:
# Optimal blend
oof_matrix = np.column_stack([xgb_result['oof'], lgb_result['oof']])

def objective_blend(w):
    pred = oof_matrix @ w
    return np.sqrt(mean_squared_error(y, pred))

res = minimize(objective_blend, [0.5, 0.5], method='SLSQP',
               bounds=[(0,1),(0,1)], 
               constraints={'type':'eq', 'fun': lambda w: sum(w)-1})

print(f'Optimal weights: XGBoost={res.x[0]:.4f}, LightGBM={res.x[1]:.4f}')
print(f'Blended RMSE: {res.fun:.6f}')

blended_oof = oof_matrix @ res.x

# Save results
results = {
    'xgboost_rmse': xgb_result['rmse'],
    'lightgbm_rmse': lgb_result['rmse'],
    'blended_rmse': res.fun,
    'xgboost_weight': res.x[0],
    'lightgbm_weight': res.x[1],
    'best_tuned_rmse': best_rmse,
    'best_tuned_params': best_params,
}

# Feature importance
print('\nTop 20 features:')
for _, row in xgb_result['fi'].head(20).iterrows():
    print(f'  {row["feature"]}: {row["importance"]:.4f}')

# Save
pd.DataFrame([results]).to_csv('/kaggle/working/training_results.csv', index=False)
xgb_result['fi'].to_csv('/kaggle/working/feature_importance.csv', index=False)
print('\nDone! Results saved to /kaggle/working/')